# REST vs. GraphQL for an AI agent

This notebook builds two LangChain agents and asks them the same questions about the conference.

- The **REST agent** can only use the four REST MCP servers, one for each service.
- The **GraphQL agent** can only use the GraphQL MCP server, which reads the schema and queries the Apollo Router.

Each run prints which tools the agent called. The notebook saves every result in the `results/` folder,
and ends with a table that compares the two agents on every question.

**Before you run it:** start the main stack and the [MCP servers](../mcp/README.md),
and log in to AWS with a profile that has Amazon Bedrock access, for example `aws sso login --profile {my-profile}`.
The first cell asks for the profile name.

In [1]:
import os

# The model, as "<provider>:<model ID>".
# This is the US cross-region inference profile for Claude Sonnet 5.5 on Amazon Bedrock.
MODEL = "bedrock_converse:us.anthropic.claude-sonnet-5-5"
os.environ.setdefault("AWS_REGION", "us-east-1")

# Ask for the AWS profile that has Amazon Bedrock access.
# Press Enter to keep the profile shown in brackets, which comes from AWS_PROFILE in your shell.
default_profile = os.environ.get("AWS_PROFILE", "")
profile = input(f"AWS profile [{default_profile}]: ").strip() or default_profile
if profile:
    os.environ["AWS_PROFILE"] = profile
print(f"Using AWS profile: {profile or 'the default credentials'}")

Using AWS profile: dev


## The agent

`run_agent` loads the tools from the MCP servers, builds a LangChain agent with them, and asks it one question.
When the agent finishes, it prints which tools the agent called and how many times,
and returns the answer and the totals for the comparison at the end.

In [2]:
import json
import logging
import time
from collections import Counter

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

# The default output limit on Bedrock is 4,096 tokens, which can cut off a long answer or a long list of tool calls.
model = init_chat_model(MODEL, max_tokens=16000)

# The MCP client logs a harmless warning each time the GraphQL MCP server answers a session close with HTTP 202.
logging.getLogger("mcp.client.streamable_http").setLevel(logging.ERROR)


async def run_agent(name, system_prompt, mcp_servers, question):
    tools = await MultiServerMCPClient(mcp_servers).get_tools()
    print(f"Tools: {', '.join(tool.name for tool in tools)}\n")

    agent = create_agent(model=model, tools=tools, system_prompt=system_prompt, name=name)

    started = time.perf_counter()
    state = await agent.ainvoke({"messages": [HumanMessage(question)]})
    seconds = time.perf_counter() - started

    model_calls = [m for m in state["messages"] if isinstance(m, AIMessage)]
    tools_called = Counter(call["name"] for m in model_calls for call in m.tool_calls)
    print("Called: " + ", ".join(f"{tool_name} × {count}" for tool_name, count in tools_called.items()))
    print(f"Took {seconds:.1f} s")
    return {
        "agent": name,
        "question": question,
        "model": MODEL,
        "answer": model_calls[-1].text if model_calls else "",
        "tool call list": [{"name": c["name"], "args": c["args"]} for m in model_calls for c in m.tool_calls],
        "tool calls": sum(len(m.tool_calls) for m in model_calls),
        "model calls": len(model_calls),
        "input tokens": sum((m.usage_metadata or {}).get("input_tokens", 0) for m in model_calls),
        "output tokens": sum((m.usage_metadata or {}).get("output_tokens", 0) for m in model_calls),
        "seconds": round(seconds, 1),
    }

## The MCP servers for each agent

The two agents differ only in which MCP servers they can use.

In [3]:
REST_MCP_SERVERS = {
    "speakers": {"transport": "streamable_http", "url": "http://localhost:5101/mcp"},
    "sessions": {"transport": "streamable_http", "url": "http://localhost:5102/mcp"},
    "rooms":    {"transport": "streamable_http", "url": "http://localhost:5103/mcp"},
    "feedback": {"transport": "streamable_http", "url": "http://localhost:5104/mcp"},
}

GRAPHQL_MCP_SERVERS = {
    "graphql": {"transport": "streamable_http", "url": "http://localhost:5100/mcp"},
}

## The shared prompt and questions

Both agents get the same system prompt and the same questions.

In [4]:
SYSTEM_PROMPT = """You answer questions about a two-day tech conference.
Today is 2026-11-12, the first day of the conference. The second day is 2026-11-13.
The venue is in London, so all times are in UTC. "This afternoon" means today from 12:00 onward.
Use only the tools you have been given to look up data. Do not guess.
Give a short, direct answer."""

QUESTIONS = [
    "For each track, which session has the highest average rating, and which room is it in?",
    "What speaker has the highest average rating and what sessions are they doing?",
    "What are the unique rooms that each track has sessions in?",
    "What sessions are happening this afternoon presented by Amara Okafor and what rooms are they in?",
    "What companies are putting on the highest-rated sessions this afternoon?"
]

## Ask both agents every question

Each result is saved in `results/<question>/<agent>.json`. A question and agent that already have a saved
result for the same model are skipped, so adding a question only runs the new one.

To run a question again, delete its folder in `results/`. To run everything again, delete `results/`.

In [5]:
import re
from pathlib import Path

RESULTS_DIR = Path("results")

AGENTS = [
    ("rest-agent", REST_MCP_SERVERS),
    ("graphql-agent", GRAPHQL_MCP_SERVERS),
]


def result_file(question, agent_name):
    """Returns the file for one question and agent, for example results/what-are-the-unique-rooms/rest-agent.json."""
    folder = re.sub(r"[^a-z0-9]+", "-", question.lower()).strip("-")[:80]
    return RESULTS_DIR / folder / f"{agent_name}.json"


results = []
for question in QUESTIONS:
    for agent_name, mcp_servers in AGENTS:
        path = result_file(question, agent_name)
        if path.exists():
            saved = json.loads(path.read_text())
            # A result from a different model, or from an older version of this notebook, runs again.
            if saved.get("model") == MODEL and "answer" in saved:
                print(f"Saved result: {agent_name} | {question}")
                results.append(saved)
                continue

        print(f"\n=== {agent_name} | {question}\n")
        result = await run_agent(agent_name, SYSTEM_PROMPT, mcp_servers, question)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(result, indent=2))
        results.append(result)

Saved result: rest-agent | For each track, which session has the highest average rating, and which room is it in?
Saved result: graphql-agent | For each track, which session has the highest average rating, and which room is it in?
Saved result: rest-agent | What speaker has the highest average rating and what sessions are they doing?
Saved result: graphql-agent | What speaker has the highest average rating and what sessions are they doing?
Saved result: rest-agent | What are the unique rooms that each track has sessions in?
Saved result: graphql-agent | What are the unique rooms that each track has sessions in?
Saved result: rest-agent | What sessions are happening this afternoon presented by Amara Okafor and what rooms are they in?
Saved result: graphql-agent | What sessions are happening this afternoon presented by Amara Okafor and what rooms are they in?
Saved result: rest-agent | What companies are putting on the highest-rated sessions this afternoon?
Saved result: graphql-agent | 

## Compare the two agents

Each row is one question.

In [6]:
import pandas as pd

METRICS = ["tool calls", "model calls", "input tokens", "output tokens", "seconds"]

table = pd.DataFrame(results).pivot(index="question", columns="agent", values=METRICS)
table = table.reindex(index=QUESTIONS, columns=["rest-agent", "graphql-agent"], level=1)
# Reshaping the table turns the counts into decimals, so change them back to whole numbers.
for metric in ["tool calls", "model calls", "input tokens", "output tokens"]:
    table[metric] = table[metric].astype(int)
table

tool calls                \
agent                                              rest-agent graphql-agent   
question                                                                      
For each track, which session has the highest a...         11             2   
What speaker has the highest average rating and...         95             3   
What are the unique rooms that each track has s...          2             2   
What sessions are happening this afternoon pres...          3             2   
What companies are putting on the highest-rated...         41             2   

                                                   model calls                \
agent                                               rest-agent graphql-agent   
question                                                                       
For each track, which session has the highest a...           4             3   
What speaker has the highest average rating and...           5             4   
What are the unique rooms that each track has s...           2             3   
What sessions are happening this afternoon pres...           3             3   
What companies are putting on the highest-rated...           3             3   

                                                   input tokens                \
agent                                                rest-agent graphql-agent   
question                                                                        
For each track, which session has the highest a...       260564         17857   
What speaker has the highest average rating and...       369304         36487   
What are the unique rooms that each track has s...        21362         13691   
What sessions are happening this afternoon pres...        30251         13670   
What companies are putting on the highest-rated...        45394         13684   

                                                   output tokens  \
agent                                                 rest-agent   
question                                                           
For each track, which session has the highest a...          7550   
What speaker has the highest average rating and...          6345   
What are the unique rooms that each track has s...          1852   
What sessions are happening this afternoon pres...           258   
What companies are putting on the highest-rated...          3091   

                                                                    seconds  \
agent                                              graphql-agent rest-agent   
question                                                                      
For each track, which session has the highest a...           689       54.5   
What speaker has the highest average rating and...           546       38.5   
What are the unique rooms that each track has s...          2379       10.5   
What sessions are happening this afternoon pres...           346        3.7   
What companies are putting on the highest-rated...           590       17.8   

                                                                  
agent                                              graphql-agent  
question                                                          
For each track, which session has the highest a...           5.8  
What speaker has the highest average rating and...           7.0  
What are the unique rooms that each track has s...          13.1  
What sessions are happening this afternoon pres...           4.5  
What companies are putting on the highest-rated...           5.4

## The answers

Check that both agents reached the same answer.

In [7]:
from IPython.display import Markdown, display

for question in QUESTIONS:
    display(Markdown(f"### {question}"))
    for result in results:
        if result["question"] == question:
            display(Markdown(f"**{result['agent']}**\n\n{result['answer']}"))

### For each track, which session has the highest average rating, and which room is it in?

**rest-agent**

Here is the top-rated session in each track, by average rating:

| Track | Session | Avg rating (reviews) | Room |
|---|---|---|---|
| AI | Lessons Learned from Prompt Engineering (session-52) | 4.43 (14) | Harbour Suite (room-07) |
| Architecture | Lessons Learned from Domain-Driven Design (session-93) | 4.75 (4) | Workshop E (room-12) |
| Data | Lessons Learned from Data Modeling (session-65) | 5.0 (3) | Workshop E (room-12) |
| Developer Experience | Testing: What We Got Wrong (session-85) | 5.0 (3) | Map Room (room-16) |
| Platform | Practical Performance (session-77) | 4.55 (29) | Atrium Stage (room-09) |

Some of these averages rest on very few reviews. If you want a more reliable ranking, the next-best sessions with more reviews are:
- **Data:** Modeling Data for a Graph (session-18), 4.67 from 12 reviews, in Riverside Theatre.
- **Developer Experience:** The First Hour of Onboarding (session-24), 4.73 from 11 reviews, in Riverside Theatre.

I checked these eight sessions with the rating-summary tool. The other sessions' averages are my own calculations from the review list. The comparisons came out clear.

**graphql-agent**

Each track's top session by average rating:

| Track | Session | Avg rating | Room |
|---|---|---|---|
| Architecture | Lessons Learned from Domain-Driven Design (session-93) | 4.75 (4 reviews) | Workshop E |
| AI | Lessons Learned from Prompt Engineering (session-52) | 4.43 (14 reviews) | Harbour Suite |
| Developer Experience | Testing: What We Got Wrong (session-85) | 5.0 (3 reviews) | Map Room |
| Platform | Practical Performance (session-77) | 4.55 (29 reviews) | Atrium Stage |
| Data | Lessons Learned from Data Modeling (session-65) | 5.0 (3 reviews) | Workshop E |

Several of these winners have only 3 or 4 reviews, so their averages are less reliable than the ones with more reviews.

### What speaker has the highest average rating and what sessions are they doing?

**rest-agent**

By raw average, two speakers tie for first at 5.0. Each of them has only one review, so the result is weak.

- **Hana Lindgren** (5.0, 1 review)
  - "Testing: What We Got Wrong", 13 Nov 11:15–12:00 UTC, room-16. She co-presents it with Kwame Fischer.
- **Rahul Walsh** (5.0, 1 review)
  - "Practical Observability", 12 Nov 14:30–15:15 UTC, room-16. He co-presents it with Hiro Gupta.

If you want the best average among speakers with a meaningful number of reviews, it is **Oskar Abara** (4.36 from 14 reviews). His sessions are:
- "API Documentation: What We Got Wrong", 13 Nov 11:15–12:00 UTC, room-09. He co-presents it with Gabriel Costa-Silva.
- "Testing in Production", 13 Nov 16:00–16:45 UTC, room-17. He co-presents it with Farah Uddin.

...

**rest-agent**

Amara Okafor has one session this afternoon:

- **"Migrating a Monolith to a Supergraph"** runs 14:30–15:15 UTC in the **Main Hall** (floor 0). Kenji Watanabe is co-presenting.

**graphql-agent**

Amara Okafor presents one session this afternoon:

- **Migrating a Monolith to a Supergraph** runs 14:30–15:15 UTC in the **Main Hall** (ground floor). She co-presents it with Kenji Watanabe.

### What companies are putting on the highest-rated sessions this afternoon?

**rest-agent**

Northwind Consulting is the standout. It's involved in several of the top-rated sessions this afternoon (12:00 onward, UTC):

| Session | Rating (reviews) | Companies |
|---|---|---|
| Modeling Data for a Graph | 4.67 (12) | **Northwind Consulting**, Tidepool |
| Practical Performance | 4.55 (29) | **Northwind Consulting**, Literacy Without Borders |
| Developer Portals at Scale | 4.37 (19) | **Northwind Consulting** |
| Gateways Compared | 4.33 (18) | Lumen Labs, Keystone Digital |

Tidepool also appears in a fifth strong session, Microservices: What We Got Wrong (4.21, 19 reviews), which it shares with Northwind.

Two sessions rated 4.5 are left out because each has only 2 reviews. They are Security: What We Got Wrong (Atlas Logistics) and Kubernetes at Scale (Code for Communities).

**graphql-agent**

Ratings for this afternoon's sessions (12:00 onward) put these sessions on top:

1. **Modeling Data for a Graph** (4.67, 12 reviews): Northwind Consulting and Tidepool.
2. **Practical Performance** (4.55, 29 reviews): Northwind Consulting and Literacy Without Borders.
3. **Security: What We Got Wrong** (4.5, 2 reviews): Atlas Logistics.
4. **Kubernetes at Scale** (4.5, 2 reviews): Code for Communities.

Northwind Consulting is the clear leader. It appears in the top two sessions, and also in Developer Portals at Scale (4.37). Tidepool is next, with one of the top two. The last two sessions have only 2 reviews each, so their ratings are less reliable.